# CS3081 — Exercise 5(c): Bayesian Network

Run the cells from top to bottom. The rejection-sampling estimate may vary slightly because it uses random samples.

In [ ]:
%pip install -q pgmpy

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD, State
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling

model = DiscreteBayesianNetwork([
    ('heatwave', 'overload'),
    ('heatwave', 'powercut'),
    ('overload', 'powercut'),
    ('powercut', 'cancel')
])

cpd_heatwave = TabularCPD('heatwave', 2, [[0.3], [0.7]],
    state_names={'heatwave': ['yes', 'no']})

cpd_overload = TabularCPD('overload', 2, [[0.6, 0.1], [0.4, 0.9]],
    evidence=['heatwave'], evidence_card=[2],
    state_names={'overload': ['yes', 'no'], 'heatwave': ['yes', 'no']})

cpd_powercut = TabularCPD('powercut', 2,
    [[0.7, 0.2, 0.5, 0.05], [0.3, 0.8, 0.5, 0.95]],
    evidence=['heatwave', 'overload'], evidence_card=[2, 2],
    state_names={'powercut': ['yes', 'no'], 'heatwave': ['yes', 'no'], 'overload': ['yes', 'no']})

cpd_cancel = TabularCPD('cancel', 2, [[0.8, 0.1], [0.2, 0.9]],
    evidence=['powercut'], evidence_card=[2],
    state_names={'cancel': ['yes', 'no'], 'powercut': ['yes', 'no']})

model.add_cpds(cpd_heatwave, cpd_overload, cpd_powercut, cpd_cancel)
print('Model is valid:', model.check_model())

In [ ]:
inference = VariableElimination(model)
exact_result = inference.query(variables=['heatwave'], evidence={'cancel': 'yes'})
exact_probability = exact_result.get_value(heatwave='yes')
print('Exact P(heatwave=yes | cancel=yes):', round(exact_probability, 4))

In [ ]:
sampler = BayesianModelSampling(model)
samples = sampler.rejection_sample(
    evidence=[State('cancel', 'yes')], size=10000, show_progress=False
)
sampling_probability = samples['heatwave'].eq('yes').mean()
print('Rejection-sampling estimate:', round(sampling_probability, 4))

The exact probability obtained using Variable Elimination is **0.5367**, while rejection sampling gives an approximate result that may differ slightly because it is based on random samples. Increasing the number of samples can generally make the sampling estimate closer to the exact probability.